# 🎤 VoxCPM2 Inference on Colab

Bu notebook, **OpenBMB/VoxCPM2** ses sentezleme modelini Colab'da (T4 GPU) çalıştırır.

**Gerekenler:**
- Colab GPU runtime (Runtime → Change runtime type → T4 GPU)
- ~10-15 dakika kurulum (sadece ilk seferde)

> ⚡ **Not:** WSL'de (RTX 4060 8GB) test edilmiş ve çalışmıştır. T4 16GB ile daha rahat çalışır.

In [ ]:
# === 1. BAĞIMLILIKLARI KUR ===
import sys, os

# torch 2.5.1 kullan (2.6.x desteklenmez)
!pip install torch==2.5.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121 -q

# flash-attn (en kritik, CUDA gerekli)
!pip install flash-attn --no-build-isolation -q

# Diğer bağımlılıklar
!pip install transformers>=4.51.0 soundfile librosa numpy tqdm xxhash pydantic torchcodec huggingface_hub -q

print("✅ Bağımlılıklar kuruldu")

In [ ]:
# === 2. KAYNAK KODU KLONLA VE KUR ===
# GitHub'dan klonla (VRAM fix dahil, hazır)
!git clone https://github.com/kstbhdr/nanovllm-voxcpm-official /content/nanovllm-voxcpm-official 2>&1 | tail -3
%cd /content/nanovllm-voxcpm-official

# Develop modunda kur
!pip install -e . --no-deps -q
print("✅ nano-vllm-voxcpm kuruldu (VRAM fix dahil)")

In [ ]:
# === 3. CUDA KONTROL ===
import torch

print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")

import flash_attn
print(f"flash-attn: OK")

from nanovllm_voxcpm import VoxCPM
print(f"VoxCPM: OK")

In [ ]:
# === 4. MODELİ İNDİR ===
from huggingface_hub import snapshot_download
import os, json

model_path = snapshot_download(repo_id="openbmb/VoxCPM2")

config_path = os.path.join(model_path, "config.json")
with open(config_path) as f:
    config = json.load(f)

print(f"Architecture: {config.get('architecture')}")
assert config.get('architecture') == 'voxcpm2'

safetensors = [f for f in os.listdir(model_path) if f.endswith('.safetensors')]
print(f"Safetensors: {len(safetensors)} dosya")

vae_ok = os.path.exists(os.path.join(model_path, "audiovae.pth"))
print(f"audiovae.pth: {'OK' if vae_ok else 'MISSING'}")
assert vae_ok, "audiovae.pth bulunamadi!"

print(f"\nModel hazir: {model_path}")

In [ ]:
# === 5. INFERENCE ÇALIŞTIR ===
import asyncio
import numpy as np
import time
from IPython.display import Audio, display
from nanovllm_voxcpm import VoxCPM


async def main():
    print("⏳ Model yükleniyor...")
    
    # T4 16GB için optimize parametreler
    server = VoxCPM.from_pretrained(
        model=model_path,
        max_num_batched_tokens=4096,
        max_num_seqs=2,
        max_model_len=2048,
        gpu_memory_utilization=0.85,
        enforce_eager=True,       # compile sorun çıkarırsa True yap
        devices=[0],
    )
    await server.wait_for_ready()
    print("✅ Model hazir!")

    model_info = await server.get_model_info()
    sample_rate = int(model_info["sample_rate"])
    print(f"🔊 Sample rate: {sample_rate} Hz")

    # Türkçe metin
    text = "Merhaba dünya! Bugün hava çok güzel, nasılsınız?"

    print(f"\n📝 Metin: {text}")
    print("🔊 Ses sentezleniyor...")

    buf = []
    start_time = time.time()
    async for data in server.generate(
        target_text=text,
        cfg_value=2,
        temperature=1.0,
    ):
        buf.append(data)

    wav = np.concatenate(buf, axis=0)
    elapsed = time.time() - start_time
    wav_duration = wav.shape[0] / sample_rate

    print(f"\n⏱️  Süre: {elapsed:.2f}s")
    print(f"🎵 Ses: {wav_duration:.2f}s")
    print(f"📊 RTF: {elapsed / wav_duration:.3f}")

    # Colab'da oynat
    display(Audio(wav, rate=sample_rate))

    await server.stop()
    print("\n✅ Tamamlandi!")


await main()

## 🛠️ Sorun Giderme

### 1. flash-attn kurulum hatası
```
!pip uninstall flash-attn -y
!pip install flash-attn==2.6.3 --no-build-isolation --no-cache-dir
```

### 2. CUDA out of memory (OOM)
T4 16GB'da nadir olur. Yine de olursa:
```
max_num_batched_tokens=2048
max_num_seqs=1
max_model_len=1024
gpu_memory_utilization=0.80
```

### 3. RuntimeError: server process exited early
Genelde VRAM yetersizliğinden. Parametreleri küçültüp tekrar dene.

### 4. flash-attn hala hata veriyorsa
```
!pip uninstall flash-attn -y
!pip install flash-attn==2.7.4 --no-build-isolation
```

### 5. WSL'de çalıştırmak için
RTX 4060 8GB'da başarıyla test edilmiştir:
- `enforce_eager=True` kullan (compile worker'ları çok RAM yer)
- `max_model_len=2048`, `max_num_seqs=1`, `gpu_memory_utilization=0.90`
- VRAM fix repo'ya dahildir, ayrıca yama gerekmez